# Notebook 3: Mock PBFT Blockchain Trust Ledger Demo
**Paper:** *Agents for Agents: An Interrogator-Based Secure Framework for Autonomous IoUT*

This notebook demonstrates the permissioned blockchain consortium governance layer:
1. PBFT consensus with 9 validators (f=2 Byzantine fault tolerance)
2. Trust delta commit workflow
3. Three-tier enforcement strategy
4. Ledger statistics and audit trail

---
> **Note:** This uses the Python mock ledger. Real deployment uses Hyperledger Fabric + `trust_ledger.go`.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'matplotlib'], check=True)
if not os.path.exists('IoUT-Interrogator-Framework'):
    subprocess.run(['git', 'clone', 'https://github.com/[username]/IoUT-Interrogator-Framework.git'], check=True)
os.chdir('IoUT-Interrogator-Framework')
sys.path.insert(0, '.')
print('Ready.')

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from blockchain.scripts.mock_ledger import PBFTConsortium, TrustRecord, EnforcementEvent
print('Imports OK.')

In [ ]:
# ── Cell 3: Initialise PBFT consortium ────────────────────────────────────
consortium = PBFTConsortium(num_validators=9, num_byzantine=0, seed=42)

print('=== PBFT Consortium Initialised ===')
print(f'  Validators:       {consortium.num_validators}')
print(f'  Fault tolerance:  f = {consortium.f} (tolerates {consortium.f} Byzantine validators)')
print(f'  Quorum required:  {consortium.quorum} of {consortium.num_validators} votes')
print(f'  O(n²) messages:   {consortium.num_validators ** 2} per consensus round')

In [ ]:
# ── Cell 4: Simulate trust deltas and enforcement ─────────────────────────
rng = np.random.default_rng(42)
tau_min = 0.65
num_agents = 10
adv_ids = {0, 1}   # Agents 0 and 1 are adversarial

commit_log = []

for interval in range(1, 11):
    for agent_id in range(num_agents):
        is_adv = agent_id in adv_ids
        score = float(rng.beta(3, 7) if is_adv else rng.beta(8, 2))

        if score < tau_min:
            record = TrustRecord(
                agent_id=f'agent_{agent_id:03d}',
                trust_score=round(score, 4),
                interval=interval,
                is_anomalous=True,
                attack_type='selective_packet_drop' if is_adv else 'transient',
                timestamp=time.time(),
                committed_by='interrogator_0',
            )
            committed = consortium.submit_trust_delta(record)

            # Tier-1: immediate local enforcement (no PBFT)
            consortium.record_enforcement(EnforcementEvent(
                agent_id=f'agent_{agent_id:03d}', tier=1,
                action='routing_exclusion', validated_by_pbft=False,
                timestamp=time.time(),
            ))
            # Tier-2: PBFT-validated throttle (if persistent)
            if interval > 3:
                consortium.record_enforcement(EnforcementEvent(
                    agent_id=f'agent_{agent_id:03d}', tier=2,
                    action='throttle', validated_by_pbft=True,
                    timestamp=time.time(),
                ))
            commit_log.append({'interval': interval, 'agent_id': agent_id,
                               'score': score, 'committed': committed})

print(f'Total anomaly events:  {len(commit_log)}')
print(f'Committed to ledger:   {sum(c["committed"] for c in commit_log)}')

In [ ]:
# ── Cell 5: Ledger statistics ─────────────────────────────────────────────
stats = consortium.ledger_stats()
print('=== Ledger Statistics ===')
for k, v in stats.items():
    print(f'  {k:<30} {v}')

# Agent 0 trust history
history = consortium.get_agent_trust_history('agent_000')
print(f'\nAgent 0 trust history ({len(history)} records):')
for h in history[:5]:
    print(f"  Interval {h['interval']:2d}: score={h['trust_score']:.4f}  "
          f"anomalous={h['is_anomalous']}")

In [ ]:
# ── Cell 6: Visualise commit volume per interval ───────────────────────────
if commit_log:
    df_log = pd.DataFrame(commit_log)
    commits_per_interval = df_log.groupby('interval')['committed'].sum()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(commits_per_interval.index, commits_per_interval.values,
           color='#2171b5', edgecolor='white', alpha=0.85)
    ax.set_xlabel('Monitoring Interval', fontsize=11)
    ax.set_ylabel('Ledger Commits', fontsize=11)
    ax.set_title('PBFT Consortium Ledger Commits per Monitoring Interval', fontsize=12)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    os.makedirs('analysis/plots', exist_ok=True)
    plt.savefig('analysis/plots/ledger_commits.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Note: Commit volume scales with anomaly frequency, not total agent count.')